In [91]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [92]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open,Volume
0,2016-05-23,99.065590,99.661865,98.972423,99.307822,15729800
1,2016-05-24,101.050064,101.171180,99.587326,99.615275,29931300
2,2016-05-25,101.776787,102.084243,101.255048,101.432070,37951700
3,2016-05-26,102.074921,102.205355,101.637030,101.879269,21885900
4,2016-05-27,102.605972,102.615291,102.121502,102.140132,19870000
...,...,...,...,...,...,...
2510,2026-05-18,705.880005,712.070007,698.849976,711.539978,49834500
2511,2026-05-19,701.530029,706.489990,695.250000,699.809998,46827500
2512,2026-05-20,713.150024,713.150024,703.789978,705.289978,36779200
2513,2026-05-21,714.510010,717.119995,706.770020,708.989990,36415400


In [93]:
def ADL(data):
    MFM = ((data['Close'] - data['Low']) - (data['High'] - data['Close'])) / (data['High'] - data['Low'])
    MFV = MFM * data['Volume']
    ADL = MFV.cumsum()
    return ADL

df['ADL'] = ADL(df)


In [94]:
def Ulcer(data, n):
    pd = ((data['ADL'] - data['ADL'].rolling(n).max()) / data['ADL'].rolling(n).max()) * 100
    sa = (pd ** 2).rolling(n).mean()
    ui = sa.apply(np.sqrt)
    return ui

df['UI'] = Ulcer(df, 14)
df['UI'] = 1 - df['UI']
df.dropna(inplace=True)
df

Price,Date,Close,High,Low,Open,Volume,ADL,UI
26,2016-06-29,99.310799,99.516316,98.301909,98.348613,31356700,3.902408e+07,-49.932267
27,2016-06-30,100.459801,100.525192,99.142629,99.478928,36300800,7.189104e+07,-50.537489
28,2016-07-01,100.964249,101.337915,100.375722,100.413089,19902800,7.633539e+07,-50.671154
29,2016-07-05,100.347672,100.571876,99.805856,100.431750,21167900,8.511216e+07,-50.747841
30,2016-07-06,101.169769,101.216473,99.553666,99.871284,24579300,1.083107e+08,-50.226372
...,...,...,...,...,...,...,...,...
2510,2026-05-18,705.880005,712.070007,698.849976,711.539978,49834500,8.781926e+09,0.925969
2511,2026-05-19,701.530029,706.489990,695.250000,699.809998,46827500,8.787426e+09,0.923316
2512,2026-05-20,713.150024,713.150024,703.789978,705.289978,36779200,8.824205e+09,0.923316
2513,2026-05-21,714.510010,717.119995,706.770020,708.989990,36415400,8.842254e+09,0.923316


In [95]:
def signal(data):
    signal = [0] * len(df)
    for i in range(2,len(df)):
        if (data.UI.iloc[i] > data.UI.iloc[i-1]) and (data.UI.iloc[i-1] < data.UI.iloc[i-2]):
            signal[i] = 1
        elif (data.UI.iloc[i-1] > 0) and (data.UI.iloc[i] < 0):
            signal[i] = 2
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)
df

Price,Date,Close,High,Low,Open,Volume,ADL,UI,signal
26,2016-06-29,99.310799,99.516316,98.301909,98.348613,31356700,3.902408e+07,-49.932267,0
27,2016-06-30,100.459801,100.525192,99.142629,99.478928,36300800,7.189104e+07,-50.537489,0
28,2016-07-01,100.964249,101.337915,100.375722,100.413089,19902800,7.633539e+07,-50.671154,0
29,2016-07-05,100.347672,100.571876,99.805856,100.431750,21167900,8.511216e+07,-50.747841,0
30,2016-07-06,101.169769,101.216473,99.553666,99.871284,24579300,1.083107e+08,-50.226372,1
...,...,...,...,...,...,...,...,...,...
2510,2026-05-18,705.880005,712.070007,698.849976,711.539978,49834500,8.781926e+09,0.925969,0
2511,2026-05-19,701.530029,706.489990,695.250000,699.809998,46827500,8.787426e+09,0.923316,0
2512,2026-05-20,713.150024,713.150024,703.789978,705.289978,36779200,8.824205e+09,0.923316,0
2513,2026-05-21,714.510010,717.119995,706.770020,708.989990,36415400,8.842254e+09,0.923316,0


In [96]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2254
1     196
2      39
Name: count, dtype: int64


(2489, 11)

In [97]:
df.set_index('Date', inplace=True)

In [111]:
bar = 2200
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                    row_heights=[0.50,0.25,0.25], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.ADL, 
                         line=dict(color='lightseagreen', width=2),
                         name='ADL'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.UI, 
                         line=dict(color='red', width=2),
                         name='Ulcer ADL'),
                         row=3, col=1)

fig.add_hline(y=1, 
              line_width=0.5, 
              line_color="grey", 
              row=3, col=1)

fig.add_hline(y=-1, 
              line_width=0.5, 
              line_color="grey", 
              row=3, col=1)

fig.add_hline(y=-2, 
              line_width=0.5, 
              line_color="grey", 
              row=3, col=1)

fig.update_layout(
    annotations=[dict(text=" Ulcer ADL Indicator ",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=1.00,
                    y=0.24,
                    showarrow=False),
                dict(text=" Accumulation Distribution Line",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=1.00,
                    y=0.54,
                    showarrow=False)])

fig.update_layout(autosize=False, width=1100, height=900, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [113]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.05*price)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                self.sell(size=0.99, sl=1.03*price, tp=0.95*price)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Start                     2016-06-29 00:00:00
End                       2026-05-22 00:00:00
Duration                   3614 days 00:00:00
Exposure Time [%]                    78.58578
Equity Final [$]                 752780.33844
Equity Peak [$]                  752780.33844
Commissions [$]                   34810.91879
Return [%]                          652.78034
Buy & Hold Return [%]               622.51959
Return (Ann.) [%]                    22.67574
Volatility (Ann.) [%]                22.18915
CAGR [%]                             15.11424
Sharpe Ratio                          1.02193
Sortino Ratio                          1.8921
Calmar Ratio                          1.29663
Alpha [%]                           339.58491
Beta                                  0.50311
Max. Drawdown [%]                   -17.48823
Avg. Drawdown [%]                    -2.68521
Max. Drawdown Duration      301 days 00:00:00
Avg. Drawdown Duration       23 days 00:00:00
# Trades                          

In [108]:
trades = stats['_trades']
best_trade = trades.sort_values("ReturnPct", ascending=False).iloc[0]
print(best_trade)

Size                            991
EntryBar                        933
ExitBar                         949
EntryPrice               167.831699
ExitPrice                195.443477
SL                              NaN
TP                       194.629843
PnL                    27183.269262
Commission               180.002849
ReturnPct                  0.163438
EntryTime       2020-03-16 00:00:00
ExitTime        2020-04-07 00:00:00
Duration           22 days 00:00:00
Tag                            None
Entry_SIGNAL                      0
Exit_SIGNAL                       0
Name: 40, dtype: object


In [114]:
trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='ADL Ulcer Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.show()

In [115]:
def randomised_trades(trades):
    cumulative_return = [0]

    for pct in trades['ReturnPct']:
        cumulative_return.append(cumulative_return[-1] + (pct * 100))

    return cumulative_return

simulations = 100
curves = []

for i in range(simulations):
    new_trades = trades.sample(frac=1).reset_index(drop=True)
    equity_curve = randomised_trades(new_trades)
    curves.append(equity_curve)

mc = go.Figure()

for equity_curve in curves:
    mc.add_trace(go.Scatter(y=equity_curve, mode='lines', opacity=0.6, showlegend=False))

mc.update_layout(
    title='ADL Ulcer Monte Carlo Simulation',
    xaxis_title='Trade Number',
    yaxis_title='% / Trade',
    template="plotly_dark",
    autosize=False,
    width=1100,
    height=700,
)

mc.update_yaxes(gridcolor="#171717")
mc.update_xaxes(gridcolor="#171717")

mc.show()